# CrisisWorld + Cortex — SFT Training & Evaluation

This notebook trains and evaluates LLM agents for outbreak crisis management.

**Two tracks:**
1. **Router SFT**: Train the Cortex Executive to route between specialist roles
2. **Single-Policy SFT**: Train a direct observation → action baseline

**Requirements:** GPU runtime (T4 is sufficient with 4-bit quantization)

In [ ]:
# Cell 1: Install dependencies
!pip install -q trl peft transformers bitsandbytes datasets accelerate huggingface_hub
!pip install -q crisis-world  # or install from source:
# !pip install -q git+https://github.com/YOUR_USERNAME/MetaFinals.git

In [ ]:
# Cell 2: Authenticate to HuggingFace
from huggingface_hub import notebook_login
notebook_login()

In [ ]:
# Cell 3: Load or create run manifest
import json

# Option A: Paste manifest from Space
MANIFEST = {
    "run_name": "demo-router-sft",
    "run_type": "router_sft",  # or "single_policy_sft"
    "base_models": {"default": "meta-llama/Llama-3.1-8B-Instruct"},
    "sft_config": {
        "learning_rate": 2e-4,
        "num_epochs": 1,
        "lora_r": 16,
        "lora_alpha": 32,
        "batch_size": 4,
        "max_seq_length": 2048
    },
    "eval_config": {
        "max_turns": 10,
        "eval_seeds": [100, 101],
        "conditions": ["flat-fat", "cortex-full", "cortex-llm"]
    },
    "quantization": "4bit"
}

print(f"Run: {MANIFEST['run_name']}")
print(f"Type: {MANIFEST['run_type']}")
print(f"Model: {MANIFEST['base_models']['default']}")

In [ ]:
# Cell 4: Generate or load dataset
from pathlib import Path

# Option A: Generate from heuristic traces
print("Generating heuristic traces for training data...")
import subprocess
subprocess.run(["python", "-m", "CrisisWorld.inference", "--agent", "cortex", "--seed", "42"], check=True)
subprocess.run(["python", "-m", "CrisisWorld.inference", "--agent", "cortex", "--seed", "43"], check=True)

# Export to SFT format
from training.data_export import export_router_sft, export_policy_sft

if MANIFEST['run_type'] == 'router_sft':
    n = export_router_sft(Path('traces'), Path('data/sft_train.jsonl'))
else:
    n = export_policy_sft(Path('traces'), Path('data/sft_train.jsonl'))

print(f'Exported {n} training examples')

In [ ]:
# Cell 5: SFT Training with TRL + PEFT + 4-bit quantization
import torch
from datasets import load_dataset
from peft import LoraConfig
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from trl import SFTTrainer, SFTConfig

sft = MANIFEST['sft_config']
base_model = MANIFEST['base_models']['default']

# Load dataset
dataset = load_dataset('json', data_files='data/sft_train.jsonl', split='train')
print(f'Dataset: {len(dataset)} examples')

# 4-bit quantization for T4 GPU
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.bfloat16,
)

# Load model + tokenizer
model = AutoModelForCausalLM.from_pretrained(
    base_model,
    quantization_config=bnb_config,
    device_map='auto',
    torch_dtype=torch.bfloat16,
)
tokenizer = AutoTokenizer.from_pretrained(base_model)
tokenizer.pad_token = tokenizer.eos_token

# LoRA config
peft_config = LoraConfig(
    r=sft['lora_r'],
    lora_alpha=sft['lora_alpha'],
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj'],
    lora_dropout=0.05,
    task_type='CAUSAL_LM',
)

# Training config
training_args = SFTConfig(
    output_dir='./adapter_output',
    num_train_epochs=sft['num_epochs'],
    per_device_train_batch_size=sft['batch_size'],
    learning_rate=sft['learning_rate'],
    gradient_accumulation_steps=4,
    warmup_ratio=0.03,
    max_seq_length=sft['max_seq_length'],
    logging_steps=10,
    save_strategy='epoch',
    fp16=False,
    bf16=True,
)

# Train!
trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset,
    peft_config=peft_config,
    tokenizer=tokenizer,
)
trainer.train()
print('Training complete!')

In [ ]:
# Cell 6: Save adapter to Hub
adapter_repo = f"{MANIFEST['run_name']}-adapter"
trainer.save_model('./adapter_output/final')
# trainer.push_to_hub(adapter_repo)  # Uncomment to push to Hub
print(f'Adapter saved to ./adapter_output/final')

In [ ]:
# Cell 7: Evaluate with CrisisWorld
# Run heuristic baseline + LLM cortex comparison
print('Running CrisisWorld evaluation...')
import subprocess
result = subprocess.run(
    ['python', '-m', 'CrisisWorld.inference', '--experiment', 'configs/experiment_llm.yaml'],
    capture_output=True, text=True
)
print(result.stdout[-500:] if result.stdout else 'No output')
if result.returncode != 0:
    print('STDERR:', result.stderr[-500:])

In [ ]:
# Cell 8: Display results
from pathlib import Path
results_path = Path('results/comparison.md')
if results_path.exists():
    from IPython.display import Markdown, display
    display(Markdown(results_path.read_text()))
else:
    print('No results file found')

In [ ]:
# Cell 9: Push results to Hub (optional)
# from training.hub import push_results
# push_results(
#     result_data={'comparison': results_path.read_text(), 'manifest': MANIFEST},
#     repo_id='your-username/crisis-world-results',
# )
print('Done! Results are in results/comparison.md')